# Profile assistant with tool calls: practice

Describe recording functions → construct schemas → decode requested arguments → execute an allowed function → append matching tool results → ask the model again.

Use this with [the walkthrough](walkthrough.md).

**Source:**

- [1_foundations/4_lab4.ipynb](../../../../../1_foundations/4_lab4.ipynb)

**Run one cell at a time:** predict → run with Shift+Enter → explain → change one thing. Select the root project .venv kernel.

**Offline adaptation:** code below uses fictional data or local libraries. It never executes the linked source, reads a credential file, sends messages, starts a UI/server, or changes the course databases. Simulations are labeled; they do not verify external integrations.

## 1. Extracted documents become prompt context

The source reads document text and places it in a prompt. This is context injection, not permanent training. A missing page extract must not break concatenation.

**Predict:** Does a page with None add text?

In [ ]:
person = "Alex Rivera"
pages = ["Alex builds Python tools.", None, "Alex teaches beginners."]
profile = "\n".join(text for text in pages if text)
prompt = f"Represent {person}. Admit missing information.\nProfile:\n{profile}"
print(prompt)
assert "None" not in prompt
assert "patent" not in profile.lower()

**Check:** No. The prompt contains only available text. A profile that never mentions patents cannot establish a patent claim.

**Change one thing:** Add a fictional skill and rebuild the prompt. Explain why the existing string does not update itself.

## 2. A tool schema describes; a dispatcher executes

The model can request a function call. Your application decodes arguments, recognizes the name, executes the function, and returns a matching tool result. No model is needed to practice this boundary.

**Predict:** Does defining the schema call the function?

In [ ]:
import json
records = []
def record_interest(email):
    records.append(email)  # In memory only; does not send email or notifications.
    return {"recorded": True}
schema = {
    "type": "function", "function": {
        "name": "record_interest", "description": "Record a supplied address",
        "parameters": {"type": "object", "properties": {"email": {"type": "string"}},
                       "required": ["email"], "additionalProperties": False},
    },
}
assert records == []
requested = {"id": "call_1", "name": "record_interest", "arguments": '{"email":"learner@example.com"}'}
allowed = {"record_interest": record_interest}  # An explicit registry of permitted functions.
if requested["name"] not in allowed:
    raise ValueError("Unknown tool")
result = allowed[requested["name"]](**json.loads(requested["arguments"]))
tool_message = {"role": "tool", "tool_call_id": requested["id"], "content": json.dumps(result)}
print(tool_message)
assert records == ["learner@example.com"]

**Check:** No. Only the explicit dispatcher call changes records. Keep the assistant tool-call message before the matching tool result in a full conversation.

**Change one thing:** Change the requested name to an unknown function and observe the error. Then add a harmless second tool to the registry.

## 3. An agent loop must make progress and stop

A loop repeats model/tool steps until a final answer or a configured limit. This fixed script of decisions shows the order without executing generated code or calling a model.

**Predict:** Which decision ends the loop?

In [ ]:
decisions = iter([
    {"kind": "tool", "name": "add", "a": 12, "b": 30},
    {"kind": "answer", "text": "The sum is 42."},
])
events = []
for turn in range(3):  # Bound the number of steps.
    decision = next(decisions)
    if decision["kind"] == "tool":
        if decision["name"] != "add":
            raise ValueError("Unknown tool")
        events.append({"tool_result": decision["a"] + decision["b"]})
    else:
        events.append({"final": decision["text"]})
        break
else:
    raise RuntimeError("No final answer within the limit")
print(events)
assert events[0]["tool_result"] == 42

**Check:** The answer decision ends it. A loop that only keeps requesting tools needs a bound and an explicit failure path.

**Change one thing:** Add another tool step before the answer. Keep a finite turn limit and explain how the stopping rule changes.

## 4. Drafting and delivering are separate steps

An agent that formats text is different from a tool that sends email or notifications. This exercise records a delivery intention in memory only.

**Predict:** Will constructing this draft contact anyone?

In [ ]:
outbox = []
draft = {"to": "learner@example.com", "subject": "Practice report", "body": "Fictional findings."}
assert outbox == []
def dry_run_send(message):
    outbox.append({"status": "dry_run", "subject": message["subject"]})
    return {"status": "dry_run"}
result = dry_run_send(draft)
print(result, outbox)
assert len(outbox) == 1

**Check:** No. Neither this draft nor this in-memory dry run sends anything. A provider send/post call is the delivery boundary.

**Change one thing:** Format the body as HTML but keep dry_run_send unchanged. Explain where real delivery would begin.

## 5. A callback receives a message and history

A UI invokes your function and displays its return value. Earlier turns are an input, not automatic model memory. This practice calls the callback directly without starting a server.

**Predict:** What changes when history contains two old messages?

In [ ]:
def chat(message, history):
    return f"You asked: {message}. Earlier messages: {len(history)}."
first = chat("Hello", [])
second = chat("Explain more", [{"role":"user", "content":"Hello"}, {"role":"assistant", "content":first}])
print(first, second, sep="\n")
assert "Earlier messages: 2" in second

**Check:** The count becomes two. Gradio wiring is gr.ChatInterface(chat, type="messages"); launch starts the server.

**Change one thing:** Make the callback return an uppercase version of the question. Compare returning a value with printing it.

## Rebuild from memory

1. Recreate the main data-flow step in a new scratch cell.
2. Add one case that should fail and identify its exception/stop condition.
3. Compare your version with the linked source cells or functions.
4. Explain where a real model or service would enter, and what it could change.

For deliberate original-source execution, see the shared [configuration guide](../../../configuration.md). All provider/transport integrations remain UNVERIFIED at runtime.